# Local / SSH starter

Select the repository `.venv` kernel after `uv sync --extra notebooks --extra deep-learning`. Paths and W&B credentials come from `.env` or the environment. Prepare `features-train.csv` and `features-valid.csv` with numeric features and binary species targets for the MLP example. The official train.csv is metadata and cannot be used directly.

Training uses `%run` so logs and progress appear directly in the notebook. Use one device here; run the standalone CLI for multiple devices.


In [ ]:
import os

from birdclef.config import LoaderConfig, RuntimeConfig, TrainConfig, get_config
from birdclef.models import BaselineExperimentConfig, BaselineModelConfig

RUN_NAME = "baseline"
DATASET_ID = "your-data-v1"
WANDB_OFFLINE = False

config = get_config(
    BaselineExperimentConfig,
    model=BaselineModelConfig(feature_columns=["feature"], target_columns=["target"]),
    tracking={"offline": WANDB_OFFLINE},
    runtime=RuntimeConfig(accelerator="auto", devices=1),
    loader=LoaderConfig(batch_size=128, val_batch_size=64, num_workers=0),
    train=TrainConfig(max_epochs=5),
)


In [ ]:
import json

if config.tracking.api_key is not None:
    os.environ["BCLEF_TRACKING__API_KEY"] = config.tracking.api_key.get_secret_value()

config_path = config.data.output_dir / f"{RUN_NAME}-config.json"
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(json.dumps(config.safe_dump(), indent=2), encoding="utf-8")
train_path = config.data.competition_dir / "features-train.csv"
valid_path = config.data.competition_dir / "features-valid.csv"


In [ ]:
%run -m birdclef.models.baseline.fit -- --config "{config_path}" --name "{RUN_NAME}" --dataset-id "{DATASET_ID}" --train "{train_path}" --valid "{valid_path}"
